In [ ]:
from typing import List

def pass_at_1_from_scores(scores: List[List[int]]) -> float:
    """
    DeepSeek-style pass@1 estimator from k samples per problem.
    """
    total, correct = 0, 0
    for row in scores:
        correct += sum(int(x) for x in row)
        total   += len(row)
    return (correct / total) if total else 0.0


In [ ]:
TEST_DIRS=[
    "data/test_codeforces.parquet",
    "data/test_lcbv5.parquet",
]
# Must match evaluation/run_inference.sh: 16K responses for Qwen3-1.7B, 8K for Qwen2.5-7B-Instruct.
MODEL="Qwen/Qwen3-1.7B"
OUTPUT_DIR="outputs/inference_results/16384/"


In [ ]:
import os
import json
import datasets
from pathlib import Path
from tqdm.auto import tqdm 
from concurrent.futures import ProcessPoolExecutor, as_completed

from verl.utils.reward_score import default_compute_score

def reward_model_processor(reward_models):
    processed_reward_models = []
    for reward_model in reward_models:
        if type(reward_model) == str:
            reward_model = json.loads(reward_model)
            reward_model["ground_truth"] = json.dumps(reward_model["ground_truth"])
        processed_reward_models.append(reward_model)
    return {"reward_model": processed_reward_models}

def load_dataset(parquet_path):
    dataframe = datasets.load_dataset("parquet", data_files=parquet_path)["train"]
    dataframe = dataframe.map(reward_model_processor,
        input_columns="reward_model",
        batched=True, batch_size=16,
        num_proc=8,
        desc="Processing reward_model"
    )
    df = dataframe.to_pandas()
    return df

def _score_prompt_bool(args):
    """
    Worker: compute a boolean pass/fail list for one prompt.
    """
    p, gts, outs, data_src = args
    scores_p = []
    for o in outs:
        try:
            scores_p.append(default_compute_score(data_src, o, gts) == 1)
        except Exception:
            # If scoring a single output fails, count it as False and continue
            scores_p.append(False)
    return p, scores_p

def compute_scores_all_mp(original_prompts, ground_truths, generations, data_src, max_workers=None):
    """
    Parallel replacement for:

        scores_all = []
        for p, gts in tqdm(zip(original_prompts, ground_truths), total=len(original_prompts)):
            po = generations[str(p)]["generations"]
            scores_p = []
            for o in po:
                scores_p.append(default_compute_score(data_src, o, gts)==1)
            scores_all.append(scores_p)

    Returns:
        scores_all: list[list[bool]] aligned with original_prompts
    """
    assert len(original_prompts) == len(ground_truths), "prompts and ground_truths length mismatch"

    # Build per-prompt job args
    jobs = []
    for p, gts in zip(original_prompts, ground_truths):
        outs = generations[str(p)]["generations"]
        jobs.append((p, gts, outs, data_src))

    if max_workers is None:
        # leave a little headroom
        max_workers = max((os.cpu_count() or 2) - 1, 1)

    scores_map = {}
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_score_prompt_bool, j) for j in jobs]
        with tqdm(total=len(jobs), desc="Scoring", dynamic_ncols=True) as pbar:
            for fut in as_completed(futures):
                try:
                    p, scores_p = fut.result()
                    scores_map[str(p)] = scores_p
                except Exception as e:
                    # Keep going even if one prompt fails completely
                    print(f"[WARN] scoring failed for a prompt: {e}")
                finally:
                    pbar.update(1)

    # Rebuild in original order
    scores_all = [scores_map.get(str(p), [False] * len(generations[str(p)]["generations"]))
                  for p in original_prompts]
    return scores_all

In [ ]:
results = {}
for parquet_path in TEST_DIRS:
    df = load_dataset(parquet_path)
    original_prompts = df["prompt"].tolist()
    ground_truths = df["reward_model"].apply(lambda x: x["ground_truth"]).values
    data_src = df["data_source"].iloc[0]
    print(data_src)

    output_filename = f"{Path(data_src).name}_{Path(MODEL).name}_generations.json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    with open(output_path, "r") as f:
        generations = json.load(f)

    scores_all = compute_scores_all_mp(original_prompts, ground_truths, generations, data_src)
    results[data_src] = 100*pass_at_1_from_scores(scores_all)
    print(f"{data_src} pass@1", results[data_src])

print(results)